# 05 — Network Features and Feedback Loops

In this notebook, I'll test whether my ML models improve by learning from new investigations each month. I'll also test whether adding network features helps detect more laundering activity.

I'll use the same data and model settings from Notebook 04, with a limit of 50 reviews per day.

# Part A — Setup

I'll connect Google Drive and check that the data, reviewed cases, scores, and models saved in Notebook 04 are available.

In [51]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/INFO320_AML")
DATA = PROJECT / "Data" / "Processed"

files = [
    "sender_day_features.parquet",
    "historical_reviews.parquet",
    "deployment_scores.parquet",
    "trained_models.joblib",
    "model_settings.json"]

for file in files:
    print(file, "Found" if (DATA / file).exists() else "MISSING")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sender_day_features.parquet Found
historical_reviews.parquet Found
deployment_scores.parquet Found
trained_models.joblib Found
model_settings.json Found


Im now loading the datasets, reviewed cases, and trained models from Notebook 04. I'll check that the number of records and laundering cases matches my previous results.

In [52]:
import pandas as pd
import joblib
import json

df = pd.read_parquet(DATA / "sender_day_features.parquet")
reviewed = pd.read_parquet(DATA / "historical_reviews.parquet")
deployment = pd.read_parquet(DATA / "deployment_scores.parquet")

models = joblib.load(DATA / "trained_models.joblib")

with open(DATA / "model_settings.json") as f:
    settings = json.load(f)

for data in [df, reviewed, deployment]:
    data["Date"] = pd.to_datetime(data["Date"])

print("Total sender-days:", len(df))
print("Historical reviews:", len(reviewed))
print("Historical laundering cases:", reviewed["label"].sum())
print("Deployment sender-days:", len(deployment))
print("Deployment laundering cases:", deployment["label"].sum())
print("Deployment days:", deployment["Date"].nunique())
print("Saved models:", list(models.keys()))


Total sender-days: 3683221
Historical reviews: 4300
Historical laundering cases: 54
Deployment sender-days: 2724979
Deployment laundering cases: 6861
Deployment days: 235
Saved models: ['logistic_regression', 'lightgbm', 'random_forest_validation', 'full_label_logistic_regression', 'full_label_lightgbm']


Im checking to make sure that the behavioral and network features are available and confirm the model settings from Notebook 04. This will make sure I'm using the same starting point for my new experiments.

In [53]:
BEHAVIOR = settings["features"]

NETWORK = [
    "max_receiver_fanin",
    "received_prior_7d"]

STAGE_A = BEHAVIOR
STAGE_B = BEHAVIOR + NETWORK

print("Behavioral features:", len(STAGE_A))
print("Network features:", len(NETWORK))
print("Total Stage B features:", len(STAGE_B))

for name, data in [("Full dataset", df), ("Historical reviews", reviewed)]:
    missing = [col for col in STAGE_B if col not in data.columns]
    print(name, "- Missing features:", missing)

print("Review capacity:", settings["review_capacity"])
print("Original review policy:", settings["review_policy"])
print("Training period:", settings["training_period"])
print("Testing period:", settings["testing_period"])


Behavioral features: 9
Network features: 2
Total Stage B features: 11
Full dataset - Missing features: []
Historical reviews - Missing features: []
Review capacity: 50
Original review policy: Top 45 rules plus 5 random from ranks 51-200
Training period: October-December 2022
Testing period: January-August 2023


# Part B - Feedback Loop Simulation

In this section I'll create a simulation where the models score sender-days and select 50 cases for investigation each day. The results of these investigations will be added to the training data.

The adaptive models will use this new information to retrain each month, while the static models will continue using their original training.

Im setting the daily review limit and the rules for selecting cases. Each model will follow the same process so I can compare their results fairly.

In [54]:
CAPACITY = 50
TOP_N = 45
RANDOM_N = 5
BAND_START = 46
BAND_END = 200
SEED = 42

print("Daily reviews:", CAPACITY)
print("Top-ranked cases:", TOP_N)
print("Randomly selected cases:", RANDOM_N)
print("Random selection range:", BAND_START, "to", BAND_END)
print("Random seed:", SEED)


Daily reviews: 50
Top-ranked cases: 45
Randomly selected cases: 5
Random selection range: 46 to 200
Random seed: 42


The models use patterns that were learned during training, such as unusual payment amounts, transaction frequency, cash activity, and international transactions, to give each sender day a risk score.

Im going to now create a function that uses these scores to rank sender-days, select the 45 highest-risk cases, and randomly choose five more from ranks 46–200. I'll use the same selection process for every model so the comparisons are fair.

In [55]:
def select_daily_reviews(day):

    ranked = day.sort_values(
        ["risk_score", "max_amount_ratio", "sender_id"],
        ascending=[False, False, True]).copy()

    ranked["rank"] = range(1, len(ranked) + 1)

    top = ranked.head(TOP_N).copy()
    top["source"] = "top45"

    band = ranked.iloc[BAND_START - 1:BAND_END]

    daily_seed = SEED + pd.Timestamp(day["Date"].iloc[0]).toordinal()
    sampled = band.sample(n=RANDOM_N, random_state=daily_seed).copy()
    sampled["source"] = "sampled"

    reviews = pd.concat([top, sampled], ignore_index=True)

    return reviews


Using the selection function I created above, I'll test it using one day of LightGBM scores to make sure it selects the correct number of cases, follows the ranking rules, and doesn't select any duplicates.

In [56]:
test_date = deployment["Date"].min()

day = deployment[deployment["Date"] == test_date].copy()
day["risk_score"] = day["lightgbm"]

selected = select_daily_reviews(day)

print("Date:", test_date.date())
print("Total selected:", len(selected))
print("Top-ranked:", (selected["source"] == "top45").sum())
print("Randomly sampled:", (selected["source"] == "sampled").sum())
print("Duplicate sender-days:", selected["sender_id"].duplicated().sum())

print("\nSelected ranks:")
print(selected[["rank", "source"]].sort_values("rank").to_string(index=False))


Date: 2023-01-01
Total selected: 50
Top-ranked: 45
Randomly sampled: 5
Duplicate sender-days: 0

Selected ranks:
 rank  source
    1   top45
    2   top45
    3   top45
    4   top45
    5   top45
    6   top45
    7   top45
    8   top45
    9   top45
   10   top45
   11   top45
   12   top45
   13   top45
   14   top45
   15   top45
   16   top45
   17   top45
   18   top45
   19   top45
   20   top45
   21   top45
   22   top45
   23   top45
   24   top45
   25   top45
   26   top45
   27   top45
   28   top45
   29   top45
   30   top45
   31   top45
   32   top45
   33   top45
   34   top45
   35   top45
   36   top45
   37   top45
   38   top45
   39   top45
   40   top45
   41   top45
   42   top45
   43   top45
   44   top45
   45   top45
   68 sampled
  151 sampled
  154 sampled
  171 sampled
  186 sampled


The function correctly selected 50 cases: the 45 highest-ranked and five randomly sampled from ranks 46–200. No duplicate cases were selected.

Now im going to prepare the starting models using the logistic regression and light GBM models saved in Notebook 04.

Each simulation will start with the same trained model, so I can compare whether retraining improves its performance.

In [57]:
from copy import deepcopy
from sklearn.base import clone

def starting_model(model_type, features):

    if features == STAGE_A:
        return deepcopy(models[model_type])

    model = clone(models[model_type])
    model.fit(reviewed[features].astype(float), reviewed["label"])

    return model


for name in ["logistic_regression", "lightgbm"]:
    model = starting_model(name, STAGE_A)
    print(name, "- Ready")


logistic_regression - Ready
lightgbm - Ready


Above, we created a function that selects cases for investigation using risk scores and we prepared our trained models.

Now we'll connect them so the models can calculate risk scores and select the 50 cases to investigate each day.

In [58]:
def run_one_day(model, day, features):

    day = day.copy()

    day["risk_score"] = model.predict_proba(
        day[features].astype(float))[:, 1]

    reviews = select_daily_reviews(day)

    return reviews


Now that i have conneted the models to the selection function ill test it using the LightGBM model on Jan 1 checking that it selects 50 cases and see how many laundering cases those investigations identfy.

In [59]:
test_date = deployment["Date"].min()

day = df[df["Date"] == test_date]
model = starting_model("lightgbm", STAGE_A)

day_reviews = run_one_day(model, day, STAGE_A)

print("Date:", test_date.date())
print("Total reviews:", len(day_reviews))
print("Top-ranked reviews:", (day_reviews["source"] == "top45").sum())
print("Sampled reviews:", (day_reviews["source"] == "sampled").sum())

print("Laundering cases found:", day_reviews["label"].sum())
print("Starting training cases:", len(reviewed))
print("Training cases after today:", len(reviewed) + len(day_reviews))


Date: 2023-01-01
Total reviews: 50
Top-ranked reviews: 45
Sampled reviews: 5
Laundering cases found: 3
Starting training cases: 4300
Training cases after today: 4350


The daily simulation worked and ran as I expected selecting 45 high risk cases and five random cases.

Above I tested the model on one day. Now im going to build a function that repeats this process across the full testing period.

Each day, the models will score sender-days, select 50 cases for investigation, and save the results. The adaptive models will retrain at the beginning of each month using all previously reviewed cases, while the static models will stay unchanged.

In [60]:
def simulate(model_type, features, adaptive, max_days=None):

    model = starting_model(model_type, features)
    training_parts = [reviewed[["Date", "label"] + features].copy()]
    daily_logs = []
    training_log = []
    version = 0

    test_days = df[df["Date"] >= "2023-01-01"].groupby("Date")

    for i, (date, day) in enumerate(test_days):

        if max_days is not None and i >= max_days:
            break

        # Retrain at the start of each new month
        if adaptive and date.day == 1 and date.month > 1:

            training_data = pd.concat(training_parts, ignore_index=True)

            assert training_data["Date"].max() < date

            model.fit(
                training_data[features].astype(float),
                training_data["label"])

            version += 1

            training_log.append({
                "Date": date,
                "Version": version,
                "Training Cases": len(training_data),
                "Positive Cases": int(training_data["label"].sum())})

        # Score and select today's investigations
        selected = run_one_day(model, day, features)
        selected["model_version"] = version

        daily_logs.append(selected[
            ["Date", "sender_id", "label", "Laundering_type",
             "rank", "source", "risk_score", "model_version"]])

        # Add today's reviewed cases to the training pool
        training_parts.append(
            selected[["Date", "label"] + features].copy())

    return (
        pd.concat(daily_logs, ignore_index=True),
        pd.DataFrame(training_log))


Now that I've created the full simulation function above I'm now going to test it before running the full actual experiments. I'll check that it selects 50 cases each day, saves the investigation results, and retrains the adaptive model on February 1 using only previously reviewed cases.

This will help me confirm the feedback loop works correctly before testing whether it improves detection in Part C.

In [61]:
# Testing January 1 through February 1
adaptive_log, adaptive_training = simulate("lightgbm", STAGE_A, True, max_days=32)
static_log, static_training = simulate("lightgbm", STAGE_A, False, max_days=32)

# Checking the daily investigations for both models
print("Days tested:", adaptive_log["Date"].nunique())
print("Reviews per day:", adaptive_log.groupby("Date").size().unique())
print("Duplicate reviews:", adaptive_log.duplicated(["Date", "sender_id"]).sum())

# Checking that both models start the same
january = adaptive_log["Date"].dt.month == 1
print("January selections match:",
      adaptive_log.loc[january, "sender_id"].tolist() ==
      static_log.loc[january, "sender_id"].tolist())

# Checking that the monthly retraining was sucessful
print("\nAdaptive retraining:")
print(adaptive_training.to_string(index=False))
print("Static retraining events:", len(static_training))
print("Model versions:", adaptive_log["model_version"].unique())


Days tested: 32
Reviews per day: [50]
Duplicate reviews: 0
January selections match: True

Adaptive retraining:
      Date  Version  Training Cases  Positive Cases
2023-02-01        1            5850              76
Static retraining events: 0
Model versions: [0 1]


The simulation worked as I expected with both versions selected the same cases throughout January, with 50 reviews per day and no duplicates.

On February 1, the adaptive model retrained using 5,850 reviewed cases, including 76 laundering cases. The static model stayed unchanged. This confirms the simulation is ready for the full experiments.

# Part C - Testing the Feedback Loop (H3)

Now that I've built and confirmed the feedback loop works correctly, I'll use it to see whether monthly retraining improves laundering detection.

I'll compare static and adaptive versions of logistic regression and LightGBM over the full 235-day testing period. Both versions will follow the same investigation process, but only the adaptive models will learn from new investigation results.

First I'm going to run the logistic regression model twice: once without retraining and once with monthly retraining.

I'll compare how many laundering cases each version catches to see whether learning from new investigations improves its performance.

In [62]:
# Runing the logistic regression without retraining
static_lr_log, static_lr_training = simulate(
    "logistic_regression", STAGE_A, adaptive=False)

# Runing the logistic regression with monthly retraining
adaptive_lr_log, adaptive_lr_training = simulate(
    "logistic_regression", STAGE_A, adaptive=True)

# Comparing the results
print("Static cases caught:", static_lr_log["label"].sum())
print("Adaptive cases caught:", adaptive_lr_log["label"].sum())

print("Days tested:", adaptive_lr_log["Date"].nunique())
print("Adaptive retraining events:", len(adaptive_lr_training))
print("Static retraining events:", len(static_lr_training))


Static cases caught: 31
Adaptive cases caught: 414
Days tested: 235
Adaptive retraining events: 7
Static retraining events: 0


This shows a promising early result for my H3. The Adaptive regression model caught 414 laundering cases compared to 31 for the static model, an increase of 383. The adaptive model retrained seven times, while the static model stayed unchanged.

These results suggest that learning from new investigations can significantly improve detection. I'll further examine the monthly results and statistical differences later in Part C.

Im going to repeat the same experiment with the LightGBM model, comparing a static version against one that retrains monthly.

This will show whether the feedback loop also improves my stronger-performing model from Notebook 04.

In [63]:
# Runing the LightGBM model without retraining
static_lgbm_log, static_lgbm_training = simulate(
    "lightgbm", STAGE_A, adaptive=False)

# Running the LightGBM model with monthly retraining
adaptive_lgbm_log, adaptive_lgbm_training = simulate(
    "lightgbm", STAGE_A, adaptive=True)

# Comparing the results
print("Static cases caught:", static_lgbm_log["label"].sum())
print("Adaptive cases caught:", adaptive_lgbm_log["label"].sum())

print("Days tested:", adaptive_lgbm_log["Date"].nunique())
print("Adaptive retraining events:", len(adaptive_lgbm_training))
print("Static retraining events:", len(static_lgbm_training))


Static cases caught: 225
Adaptive cases caught: 455
Days tested: 235
Adaptive retraining events: 7
Static retraining events: 0


Adaptive LightGBM caught 455 laundering cases compared to 225 for the static model, an increase of 230.

At first glance the regression model showed the larger improvement, gaining 383 cases, but adaptive LightGBM still caught the most overall (455 vs. 414). Both models performed better with monthly retraining, which I'll examine further in the next sections.

Now that I've tested both models, I'll compare how many laundering cases each version caught, along with their recall and precision. This will show how much monthly retraining improved their overall performance.

In [64]:
results = []

for name, log in [
    ("Static Logistic Regression", static_lr_log),
    ("Adaptive Logistic Regression", adaptive_lr_log),
    ("Static LightGBM", static_lgbm_log),
    ("Adaptive LightGBM", adaptive_lgbm_log)]:
    caught = int(log["label"].sum())

    results.append({
        "Model": name,
        "Caught": caught,
        "Recall (%)": round(caught / deployment["label"].sum() * 100, 2),
        "Precision (%)": round(caught / len(log) * 100, 2)})

overall_results = pd.DataFrame(results)

display(overall_results)


,Model,Caught,Recall (%),Precision (%)
0,Static Logistic Regression,31,0.45,0.26
1,Adaptive Logistic Regression,414,6.03,3.52
2,Static LightGBM,225,3.28,1.91
3,Adaptive LightGBM,455,6.63,3.87


The overall results show that both models preformed much better with monthly retraining.

However, these totals don't show when the improvements occurred or whether they continued over time. I'll look at the monthly results next to better understand how performance changed after retraining.

To help me see when the adaptive models began outperforming their static versions after retraining I'll compare how many laundering cases each model caught each month and calculate their monthly recall.

In [65]:
logs = {
    "Static LR": static_lr_log,
    "Adaptive LR": adaptive_lr_log,
    "Static LightGBM": static_lgbm_log,
    "Adaptive LightGBM": adaptive_lgbm_log}

monthly_caught = pd.DataFrame()

for name, log in logs.items():
    monthly_caught[name] = log.groupby(
        log["Date"].dt.to_period("M"))["label"].sum()

monthly_total = deployment.groupby(
    deployment["Date"].dt.to_period("M"))["label"].sum()

monthly_recall = (monthly_caught.div(monthly_total, axis=0) * 100).round(2)

print("Monthly laundering cases caught:")
display(monthly_caught)

print("Monthly recall (%):")
display(monthly_recall)


Monthly laundering cases caught:


,Static LR,Adaptive LR,Static LightGBM,Adaptive LightGBM
Date,,,,
2023-01,6,6,22,22
2023-02,4,45,19,34
2023-03,5,40,24,78
2023-04,2,51,41,22
2023-05,0,68,21,46
2023-06,6,75,43,93
2023-07,6,75,37,81
2023-08,2,54,18,79


Monthly recall (%):


,Static LR,Adaptive LR,Static LightGBM,Adaptive LightGBM
Date,,,,
2023-01,0.68,0.68,2.50,2.50
2023-02,0.45,5.07,2.14,3.83
2023-03,0.59,4.71,2.83,9.19
2023-04,0.23,5.74,4.62,2.48
2023-05,0.00,8.62,2.66,5.83
2023-06,0.66,8.21,4.71,10.19
2023-07,0.66,8.30,4.09,8.96
2023-08,0.27,7.20,2.40,10.53


These results give us a much clearer picture of how the feedback loop performed over time.

Both models started with identical results in January and began showing improvements after their first retraining in February.

Adaptive logistic regression outperformed its static version every month after retraining. LightGBM also improved in most months, although its adaptive version performed worse in April.

The adaptive models caught more laundering cases overall, but I want to check how reliable these improvements are.

Im use going to use the same bootstrap method from Notebook 04, which repeatedly samples the daily results 1,000 times to see how much the difference between static and adaptive models could vary.

This will give me a 95% confidence interval for each model's improvement and help me determine whether the results provide stronger evidence that monthly retraining improves detection.

In [66]:
import numpy as np

# Counting laundering cases caught each day
daily = pd.DataFrame({
    "Static LR": static_lr_log.groupby("Date")["label"].sum(),
    "Adaptive LR": adaptive_lr_log.groupby("Date")["label"].sum(),
    "Static LightGBM": static_lgbm_log.groupby("Date")["label"].sum(),
    "Adaptive LightGBM": adaptive_lgbm_log.groupby("Date")["label"].sum()})

rng = np.random.default_rng(42)

# Comparing the adaptive against static for each model
for static, adaptive in [
    ("Static LR", "Adaptive LR"),
    ("Static LightGBM", "Adaptive LightGBM")]:
    differences = (daily[adaptive] - daily[static]).to_numpy()

    samples = [
        rng.choice(differences, size=len(differences), replace=True).sum()
        for _ in range(1000)]

    lower, upper = np.percentile(samples, [2.5, 97.5])

    print(adaptive)
    print("Additional cases caught:", differences.sum())
    print("95% bootstrap interval:", round(lower), "to", round(upper))
    print()


Adaptive LR
Additional cases caught: 383
95% bootstrap interval: 337 to 429

Adaptive LightGBM
Additional cases caught: 230
95% bootstrap interval: 183 to 278



As expected both models had positive bootstrap intervals, providing stronger evidence that monthly retraining improved detection in this simulation.

And again the regression model showed the larger improvement, with an estimated gain of 337 to 429 cases, compared to 183 to 278 for LightGBM.

Now that I've compared the tow models performances and tested their results, Im going to look at how their training data grew over time.

I'll track how many reviewed cases and positive laundering examples each adaptive model had available at the start of each month and I will compare that with how many laundering cases it caught in that month.

 This will help me understand how detection changed as the models learned from more investigations.

In [67]:
# Starting training data for January
initial = pd.DataFrame([{
    "Date": pd.Timestamp("2023-01-01"),
    "Training Cases": len(reviewed),
    "Positive Cases": int(reviewed["label"].sum())}])

# Combine the training records for both adaptive models
learning_curve = pd.concat([
    pd.concat([initial, adaptive_lr_training], ignore_index=True)
      .assign(Model="Logistic Regression"),

    pd.concat([initial, adaptive_lgbm_training], ignore_index=True)
      .assign(Model="LightGBM")
], ignore_index=True)

learning_curve["Month"] = learning_curve["Date"].dt.to_period("M")

# Match each month's training data with its detections
learning_curve["Caught"] = [
    monthly_caught.loc[
        month,
        "Adaptive LR" if model == "Logistic Regression" else "Adaptive LightGBM"
    ]
    for month, model in zip(learning_curve["Month"], learning_curve["Model"])]

display(learning_curve[
    ["Month", "Model", "Training Cases", "Positive Cases", "Caught"]
])


,Month,Model,Training Cases,Positive Cases,Caught
0,2023-01,Logistic Regression,4300,54,6
1,2023-02,Logistic Regression,5850,60,45
2,2023-03,Logistic Regression,7250,105,40
3,2023-04,Logistic Regression,8800,145,51
4,2023-05,Logistic Regression,10300,196,68
5,2023-06,Logistic Regression,11850,264,75
6,2023-07,Logistic Regression,13350,339,75
7,2023-08,Logistic Regression,14900,414,54
8,2023-01,LightGBM,4300,54,22
9,2023-02,LightGBM,5850,76,34


In training cases both models were identical both models started with 4,300 training cases, including 54 laundering cases and by August, each had 14,900 training cases because they continued collecting 50 new investigation results every day as expected.

However,the regression model had 414 positive training examples compared to 430 for LightGBM. This difference is because each model selected different cases for investigation. Even though they reviewed the same number of cases, LightGBM found 16 more laundering cases through July, giving it more positive examples to learn from. So the feedback loop affects not only how the models learn, but also what information they get to learn from. Each model's decisions influence its future training data.


## Part C — Summary

The results support H3. Monthly retraining did improve detection for both models, with logistic regression improving the most by catching 383 additional laundering cases and LightGBM catching 230 more compared to their static versions.

Both models also showed positive bootstrap intervals, although performance didn't improve every month. Overall, the feedback loop helped the models detect more laundering as they learned from new investigations.

# Part D — Adaptive Models vs. Rules

Now that I've tested the feedback loop against the static models, I'll compare the adaptive models against the original rules based system from Notebook 04.

I want to see whether the improvements from monthly retraining would actually help a bank detect more laundering with the same daily investigation limit.

I'm starting by checking the rules based results saved in Notebook 04. This will give me a starting point to compare against the adaptive models.

In [68]:
rules_top50_reviews = deployment[deployment["rules_top50"]].copy()

print("Days tested:", rules_top50_reviews["Date"].nunique())
print("Total reviews:", len(rules_top50_reviews))
print("Reviews per day:", rules_top50_reviews.groupby("Date").size().unique())
print("Laundering cases caught:", rules_top50_reviews["label"].sum())


Days tested: 235
Total reviews: 11750
Reviews per day: [50]
Laundering cases caught: 168


The results above matched what I expected from Notebook 04, with the rules catching 168 laundering cases.

Now I'll apply the same selection process used by the models in Part C, where 45 of the highest-risk cases and five randomly sampled cases are reviewed each day.

In [69]:
daily_reviews = []

for date, day in deployment.groupby("Date"):
    day = day.copy()
    day["risk_score"] = day["rules"]
    daily_reviews.append(select_daily_reviews(day))

rules_45_5_log = pd.concat(daily_reviews, ignore_index=True)

print("Days tested:", rules_45_5_log["Date"].nunique())
print("Reviews per day:", rules_45_5_log.groupby("Date").size().unique())
print("Top-ranked reviews:", (rules_45_5_log["source"] == "top45").sum())
print("Sampled reviews:", (rules_45_5_log["source"] == "sampled").sum())
print("Laundering cases caught:", rules_45_5_log["label"].sum())


Days tested: 235
Reviews per day: [50]
Top-ranked reviews: 10575
Sampled reviews: 1175
Laundering cases caught: 163


As expected the rules selected 50 cases each day with 45 highest ranked cases and five randomly sampled cases. This approach only caught 163 laundering cases which is 5 fewer than the original top-50 method.

The difference comes from replacing five guaranteed high risk reviews with 5 randomly sampled cases from a larger group. Although this does slightly reduced detection, the random reviews help investigate activity the rules might normally overlook.

Now that the rules and adaptive models all use the same review process, I can compare their overall performance.

Im now going to compare the rules based results with the adaptive logistic regression and LightGBM models results to see which system caught the most laundering cases.

In [70]:
results = []

systems = {
    "Rules (Top 50)": rules_top50_reviews,
    "Rules (45 + 5)": rules_45_5_log,
    "Adaptive Logistic Regression": adaptive_lr_log,
    "Adaptive LightGBM": adaptive_lgbm_log}

for name, log in systems.items():
    caught = int(log["label"].sum())

    results.append({
        "System": name,
        "Caught": caught,
        "Recall (%)": round(caught / deployment["label"].sum() * 100, 2),
        "Precision (%)": round(caught / len(log) * 100, 2)})

business_results = pd.DataFrame(results)
display(business_results)


,System,Caught,Recall (%),Precision (%)
0,Rules (Top 50),168,2.45,1.43
1,Rules (45 + 5),163,2.38,1.39
2,Adaptive Logistic Regression,414,6.03,3.52
3,Adaptive LightGBM,455,6.63,3.87


The overall results confirm what was expected and show that both adaptive models outperform the rules. Next, I'll compare their monthly results to see when these differences started and whether they continued over time.

Im going to compare how many laundering cases the rules and adaptive models caught each month. What im most interested in is finding out when the adaptive logistic regression first outperformed the rules because in Notebook 04 the regression model performed much worse the the rules based model.

I'll also check whether the adaptive models continued outperforming the rules in later months.

In [71]:
monthly_business = pd.DataFrame({
    name: log.groupby(log["Date"].dt.to_period("M"))["label"].sum()
    for name, log in systems.items()})

display(monthly_business)

better = monthly_business[
    monthly_business["Adaptive Logistic Regression"] >
    monthly_business["Rules (45 + 5)"]]

print("First month adaptive LR beat the rules:",
      better.index[0] if len(better) else "None")


,Rules (Top 50),Rules (45 + 5),Adaptive Logistic Regression,Adaptive LightGBM
Date,,,,
2023-01,15,15,6,22
2023-02,22,19,45,34
2023-03,18,18,40,78
2023-04,22,20,51,22
2023-05,17,17,68,46
2023-06,29,29,75,93
2023-07,33,32,75,81
2023-08,12,13,54,79


First month adaptive LR beat the rules: 2023-02


The adaptive regression model begain outpreforming the rules based model immediately after its first retraining, and continued doing so every month afterward.  

These results show that the adaptive models performed better than the rules not just overall, but consistently across most of the testing period.

 Just like in Part C I will use the bootstrap method to check how reliable the differences are.

This time, instead of comparing the adaptive models against their static versions, I'm going to compare them against the rules based system using the same 45 + 5 review process.

In [72]:
daily_business = pd.DataFrame({
    name: log.groupby("Date")["label"].sum()
    for name, log in systems.items()})

rng = np.random.default_rng(42)

for name in ["Adaptive Logistic Regression", "Adaptive LightGBM"]:
    differences = (daily_business[name] - daily_business["Rules (45 + 5)"]).to_numpy()

    samples = [
        rng.choice(differences, size=len(differences), replace=True).sum()
        for _ in range(1000)]

    lower, upper = np.percentile(samples, [2.5, 97.5])

    print(name)
    print("Additional cases caught:", differences.sum())
    print("95% bootstrap interval:", round(lower), "to", round(upper))
    print()


Adaptive Logistic Regression
Additional cases caught: 251
95% bootstrap interval: 207 to 295

Adaptive LightGBM
Additional cases caught: 292
95% bootstrap interval: 238 to 344



Both adaptive models outperformed the rules, and their bootstrap intervals were both entirely positive. This gives me more confidence that the differences are meaningful and not just due to random variation in the daily results.

# Part E - Testing Network Features (H2)



Now that I've tested the feedback loop and compared the adaptive models against the rules, I'm going to investigate whether adding network features improves detection.

I'll first look at which laundering types the adaptive models already caught, then add the two network features and compare the results to see whether they help detect more laundering, especially schemes involving multiple accounts.

Before adding network features, Im going to check which laundering types the adaptive models already caught using the original nine behavioral features and compare with the static.

In Notebook 04, LightGBM caught mostly behavioral change laundering. So I wam most interested in seeing whether monthly retraining helped it detect other types and whether logistic regression caught different ones.

In [73]:
total_types = deployment.loc[
    deployment["label"] == 1, "Laundering_type"].value_counts()

typology_before = pd.DataFrame({"Total Cases": total_types})

for name, log in {
    "Static LR": static_lr_log,
    "Adaptive LR": adaptive_lr_log,
    "Static LightGBM": static_lgbm_log,
    "Adaptive LightGBM": adaptive_lgbm_log}.items():
    caught = log.loc[log["label"] == 1, "Laundering_type"].value_counts()
    typology_before[name] = caught.reindex(total_types.index, fill_value=0)

display(typology_before)


,Total Cases,Static LR,Adaptive LR,Static LightGBM,Adaptive LightGBM
Laundering_type,,,,,
structuring,1373,17,62,2,31
cash_withdrawal,801,0,60,2,128
deposit_send,732,1,62,1,37
smurfing,710,0,2,1,1
layered_fan_in,528,0,22,2,13
layered_fan_out,322,3,7,1,8
stacked_bipartite,322,0,3,0,5
cycle,312,2,10,0,19
scatter_gather,264,1,10,2,10


This to me is one of the most interesting findings so far. LightGBM significatly changed after monthly retraining. Before retraining, 206 of its 225 detections (91.6%) came from the two behavioral-change types. After retraining, only 177 of its 455 detections (38.9%) came from those types and it improved detection in almost every other laundering type. This suggests the feedback loop helped LightGBM detect a wider variety of laundering.

Another takeaway is that the models have different strengths. Adaptive logistic regression caught more structuring and deposit send cases, while LightGBM caught more cash withdrawal cases. This makes me interested in whether combining the two models could improve detection even further.

However, both models still did struggle with certain types of laundering. For example out of 710 smurfing cases, logistic regression only caught 2 and LightGBM caught 1. I'll now test whether adding network features helps the models detect more of these missed cases.

I'll now repeat the same four simulations as above using all 11 features instead of just the nine behavioural features. Everything else will stay the same, including the daily review limit and monthly retraining process.

I'll compare the results against the original models to see how much the two additional network features affect detection.

This will let me test whether the extra network information helps the models detect more laundering.

In [74]:
stage_a_logs = {
    "Static LR": static_lr_log,
    "Adaptive LR": adaptive_lr_log,
    "Static LightGBM": static_lgbm_log,
    "Adaptive LightGBM": adaptive_lgbm_log}

stage_b_logs = {}
stage_b_training = {}

for name, model, adaptive in [
    ("Static LR", "logistic_regression", False),
    ("Adaptive LR", "logistic_regression", True),
    ("Static LightGBM", "lightgbm", False),
    ("Adaptive LightGBM", "lightgbm", True)]:
    stage_b_logs[name], stage_b_training[name] = simulate(
        model, STAGE_B, adaptive)

results = []

for name in stage_a_logs:
    a = int(stage_a_logs[name]["label"].sum())
    b = int(stage_b_logs[name]["label"].sum())
    results.append([name, a, b, b - a])

stage_comparison = pd.DataFrame(
    results,
    columns=["Model", "Stage A", "Stage B", "Difference"])

display(stage_comparison.set_index("Model"))


,Stage A,Stage B,Difference
Model,,,
Static LR,31,33,2
Adaptive LR,414,451,37
Static LightGBM,225,254,29
Adaptive LightGBM,455,716,261


Adding the two network features improved detection across all four models, but the biggest improvement was by far with the adaptive LightGBM model.

It went from catching 455 cases to 716, an increase of 261 cases (57.4%). The other three models also improved, but by much smaller amounts.

Next, I want to see which laundering types account for these additional detections.

### E4 — Comparing Laundering Types

I'll compare which laundering types the adaptive models caught before and after adding network features. I'll pay particular attention to the nine network-related types identified in Notebook 03.

This will help me determine whether the extra features improve detection of network schemes or if it affect other types of laundering.

In [75]:
network_types = [
    "cycle", "fan_in", "fan_out",
    "layered_fan_in", "layered_fan_out",
    "bipartite", "stacked_bipartite",
    "gather_scatter", "scatter_gather"]

typology_comparison = pd.DataFrame({"Total Cases": total_types})

for name, log in {
    "Stage A LR": adaptive_lr_log,
    "Stage B LR": stage_b_logs["Adaptive LR"],
    "Stage A LightGBM": adaptive_lgbm_log,
    "Stage B LightGBM": stage_b_logs["Adaptive LightGBM"]}.items():
    caught = log.loc[log["label"] == 1, "Laundering_type"].value_counts()
    typology_comparison[name] = caught.reindex(total_types.index, fill_value=0)

typology_comparison["LR Change"] = (
    typology_comparison["Stage B LR"] - typology_comparison["Stage A LR"])

typology_comparison["LightGBM Change"] = (
    typology_comparison["Stage B LightGBM"] - typology_comparison["Stage A LightGBM"])

print("Network-related laundering types:")
display(typology_comparison.loc[network_types])

print("All laundering types:")
display(typology_comparison)


Network-related laundering types:


,Total Cases,Stage A LR,Stage B LR,Stage A LightGBM,Stage B LightGBM,LR Change,LightGBM Change
Laundering_type,,,,,,,
cycle,312,10,29,19,48,19,29
fan_in,250,8,9,8,9,1,1
fan_out,128,2,1,5,3,-1,-2
layered_fan_in,528,22,24,13,17,2,4
layered_fan_out,322,7,7,8,13,0,5
bipartite,257,4,4,6,7,0,1
stacked_bipartite,322,3,2,5,7,-1,2
gather_scatter,229,1,3,5,17,2,12
scatter_gather,264,10,13,10,11,3,1


All laundering types:


,Total Cases,Stage A LR,Stage B LR,Stage A LightGBM,Stage B LightGBM,LR Change,LightGBM Change
Laundering_type,,,,,,,
structuring,1373,62,64,31,62,2,31
cash_withdrawal,801,60,27,128,115,-33,-13
deposit_send,732,62,70,37,85,8,48
smurfing,710,2,2,1,5,0,4
layered_fan_in,528,22,24,13,17,2,4
layered_fan_out,322,7,7,8,13,0,5
stacked_bipartite,322,3,2,5,7,-1,2
cycle,312,10,29,19,48,19,29
scatter_gather,264,10,13,10,11,3,1


Adding the network features helped both models catch more network-related laundering. Logistic regression caught 25 additional cases, while LightGBM caught 53 more. Both models improved the most in detecting cycles.

What I found interesting was that only 53 of LightGBM's 261 additional detections were network-related. Another 132 came from behavioral-change cases. This suggests the network features helped with more than just network-related laundering.


I'm going to now use the same bootstrap method as before to see how reliable the improvements were after adding the two network features.

In [76]:
rng = np.random.default_rng(42)

for name in stage_a_logs:
    a = stage_a_logs[name].groupby("Date")["label"].sum()
    b = stage_b_logs[name].groupby("Date")["label"].sum()

    differences = (b - a).to_numpy()

    samples = [
        rng.choice(differences, size=len(differences), replace=True).sum()
        for _ in range(1000)]

    lower, upper = np.percentile(samples, [2.5, 97.5])

    print(name)
    print("Additional cases caught:", int(differences.sum()))
    print("95% bootstrap interval:", round(lower), "to", round(upper))
    print()


Static LR
Additional cases caught: 2
95% bootstrap interval: -13 to 17

Adaptive LR
Additional cases caught: 37
95% bootstrap interval: 6 to 67

Static LightGBM
Additional cases caught: 29
95% bootstrap interval: 11 to 47

Adaptive LightGBM
Additional cases caught: 261
95% bootstrap interval: 212 to 308



The bootstrap results showed that adding the network features improved detection for three of the four models with the adaptive LightGBM model having the largest improvement, with 261 additional cases caught and a 95% bootstrap interval of 212 to 308.

Overall, these results show that adding network features does improve detection, especially for adaptive LightGBM model.

Overall, my results support H2. Adding network features improves detection for most models, with adaptive LightGBM showing the biggest improvement.

However, with the addition of network features some laundering types actually had fewer detections, and smurfing remained difficult to catch.

Based on these results, adaptive LightGBM with all 11 features is my strongest model so far.

# Part F - Operational Impact and Model Strategy

Now that I've finished testing the models, I want to look at what these results could mean for a bank by examining how the strongest model could help reduce investigator workload and save time.

I also want to test whether combining the models could improve detection,

In Part E, I found that te logistic regression model and the LightGBM model were better at detecting different types of laundering. Now I want to see how many individual cases they both caught and how many were only caught by one model.

This will help me decide whether combining the two models is worth testing.

In [77]:
lr = stage_b_logs["Adaptive LR"]
lgbm = stage_b_logs["Adaptive LightGBM"]

lr_cases = lr.loc[lr["label"] == 1, ["Date", "sender_id"]]
lgbm_cases = lgbm.loc[lgbm["label"] == 1, ["Date", "sender_id"]]

both = lr_cases.merge(lgbm_cases, on=["Date", "sender_id"])

print("Caught by both:", len(both))
print("Only Logistic Regression:", len(lr_cases) - len(both))
print("Only LightGBM:", len(lgbm_cases) - len(both))
print("Total unique cases:", len(lr_cases) + len(lgbm_cases) - len(both))


Caught by both: 312
Only Logistic Regression: 139
Only LightGBM: 404
Total unique cases: 855


Both the models caught quite a few different cases with the regression model catching 139 cases that LightGBM missed, and LightGBM catching 404 that the regression model missed and both models caught 312 of the same cases.

Together, they caught 855 different cases, which makes me interested in testing whether combining them could improve detection.

Above I found that the regression model caught 139 laundering cases that LightGBM missed and made me wonder whether using both models together could improve detection.

Rather than building a completely new model, I'll test a simple approach where both models help choose which cases the bank investigates, using January as a quick test before deciding whether a larger experiment is worthwhile.

I'll keep the same limit of 50 reviews per day, but divide them differently:

35 reviews: The highest risk cases selected by LightGBM.

10 reviews: Additional cases selected by the regression model that LightGBM hasn't already chosen.

5 reviews: Random cases selected from LightGBM's ranks 46–200 for exploration.

I'll compare this approach against LightGBM's original 45 + 5 review process to see whether giving logistic regression some of the review capacity improves detection.

In [79]:
lr_model = starting_model("logistic_regression", STAGE_B)
lgbm_model = starting_model("lightgbm", STAGE_B)

january = df[df["Date"].between("2023-01-01", "2023-01-31")]
caught = 0

for date, day in january.groupby("Date"):
    day = day.copy()
    X = day[STAGE_B].astype(float)

    # Scoring every sender day using both models
    day["LR"] = lr_model.predict_proba(X)[:, 1]
    day["LightGBM"] = lgbm_model.predict_proba(X)[:, 1]

    # Ranking the cases by LightGBM risk score
    ranked = day.sort_values(
        ["LightGBM", "max_amount_ratio", "sender_id"],
        ascending=[False, False, True])

    # Selecting the 35 cases from LightGBM and 10 more from LR
    top35 = ranked.head(35)
    top10 = day.drop(top35.index).nlargest(10, "LR")

    # Randomly selecting 5 more cases for exploration
    random5 = ranked.iloc[45:200].drop(
        top10.index, errors="ignore").sample(5, random_state=42 + date.toordinal())

    # Counting the laundering cases detected that day
    reviews = pd.concat([top35, top10, random5])
    caught += reviews["label"].sum()

# Compareing against LightGBM's original January results
baseline = stage_b_logs["Static LightGBM"]
baseline = baseline[baseline["Date"].between("2023-01-01", "2023-01-31")]

print("LightGBM alone:", int(baseline["label"].sum()))
print("Combined approach:", int(caught))


LightGBM alone: 24
Combined approach: 25


The combined approach caught 25 laundering cases and LightGBM alone caught 24 only 1 extra case.

Since this was only a quick test using January data, I can't say for certain whether combining the models would help over the full testing period. However,I  don't think this small improvement is worth the added complexity right now, so I'll continue using adaptive LightGBM as my main model.

In Notebook 04 I tested the static models to see if they could catch as much laundering as the rules while requiring fewer investigations. I will now repeat that analysis using my strongest model the adaptive LightGBM with all 11 features.

For a bank, reducing unnecessary investigations could save analysts time and lower compliance costs. I'll test different daily review limits to see how much the bank could reduce its workload without catching fewer laundering cases than the rules.



In [82]:
def test_capacity(limit):
    model = starting_model("lightgbm", STAGE_B)
    history = [reviewed[["Date", "label"] + STAGE_B].copy()]
    caught = 0

    for date, day in df[df["Date"] >= "2023-01-01"].groupby("Date"):

        # Retraining at the beginning of each month
        if date.day == 1 and date.month > 1:
            training = pd.concat(history)
            model.fit(training[STAGE_B].astype(float), training["label"])

        # Scoring and ranking today's cases
        day = day.copy()
        day["risk_score"] = model.predict_proba(day[STAGE_B].astype(float))[:, 1]

        ranked = day.sort_values(
            ["risk_score", "max_amount_ratio", "sender_id"],
            ascending=[False, False, True])

        # Selecting the highest risk cases and 5 random cases
        top = ranked.head(limit - 5)
        random5 = ranked.iloc[limit - 5:200].sample(
            5, random_state=42 + date.toordinal())

        selected = pd.concat([top, random5])

        caught += selected["label"].sum()
        history.append(selected[["Date", "label"] + STAGE_B])

    return int(caught)

results = []

for limit in [5,6,7,8,9,10, 15, 20, 30, 40, 50]:
    caught = test_capacity(limit)
    results.append([limit, limit * 235, caught])

capacity_results = pd.DataFrame(
    results,
    columns=["Daily Reviews", "Total Reviews", "Cases Caught"])

display(capacity_results)


,Daily Reviews,Total Reviews,Cases Caught
0,5,1175,12
1,6,1410,90
2,7,1645,110
3,8,1880,170
4,9,2115,199
5,10,2350,236
6,15,3525,339
7,20,4700,377
8,30,7050,465
9,40,9400,589


The results show that adaptive LightGBM could significantly reduce the bank's investigation workload. At just 8 reviews per day, it caught 170 laundering cases whauch is 7 cases above the rules based model when its testing 50 cases per day.

However, I think 10 reviews per day is a better option. At that limit, LightGBM caught 236 cases which is 73 more than the rules, while reducing investigations by 80%.

Overall these results show how our LightGBM model could help banks investigate fewer cases while still improving detection.

Im now going to calculate how many investigations could be avoided and estimate the time and money saved using 37 minutes per review as my main assumption and for labor costs I will assume $ 40 per hour

The 37-minute estimate is the same benchmark I used in Notebook 04. It comes from a June 2025 Boston Consulting Group (BCG) report

I'll also calculate 15, 30, and 60 minutes per review to see how different assumptions affect the results.

In [84]:
days = 235
rules_reviews = 50 * days
model_reviews = 10 * days

reviews_avoided = rules_reviews - model_reviews
hourly_cost = 40

minutes = [15, 30, 37, 60]

time_savings = pd.DataFrame({
    "Minutes per Review": minutes})

time_savings["Reviews Avoided"] = reviews_avoided

time_savings["Hours Saved"] = (
    reviews_avoided * time_savings["Minutes per Review"] / 60).round(1)

time_savings["Labor Value ($)"] = (
    time_savings["Hours Saved"] * hourly_cost).round(0)

display(time_savings)


,Minutes per Review,Reviews Avoided,Hours Saved,Labor Value ($)
0,15,9400,2350.0,94000.0
1,30,9400,4700.0,188000.0
2,37,9400,5796.7,231868.0
3,60,9400,9400.0,376000.0


Reducing investigations down from 50 to 10 cases per day would eliminate 9,400 reviews over the testing period.

When using my main assumption of 37 minutes per review it would save about 5,797 investigator hours worth approximately 232,000 dollars at the assumed labor cost of 40 dollars per hour but depending on how long each review takes, the estimated time saved ranges from 2,350 to 9,400 hours or 94,000 to 376,000 dollars.

## Part G — Final Results and Tableau Exports

Now that I've finished testing all the models and calculating all their potential business impacts, I will summarize the results of this notbook and prepare and export results for my Tableau dashboard and final presentation.

## Part G — Final Summary

The results show that both the monthly retraining and adding the network features improved laundering detection, with adaptive LightGBM using all 11 features performing the best of all the models. It caught 716 laundering cases compared with 163 for the rules-based system at the same daily review limit.

also using the adaptive LightGBM model the bank could potentially reduce investigations from 50 to 10 per day while still catching more laundering than the rules based model. Based on my assumptions, this could free up approximately 5,797 investigator hours, worth about $232,000 in labor time.

Overall, I would recommend adaptive LightGBM for further testing, while recognizing that these results are based on synthetic data and would need to be validated before use in a real bank.

# Part G — Tableau Exports

I'll now organize and export my results as CSV files for Tableau and my presentation.

I'll start with the overall model comparison, then export the monthly results, laundering types, and estimated business impact.

Creating and exporting one comparioson table for all nine models

In [85]:
# Creating the Tableau folder
TABLEAU = PROJECT / "Tableau"
TABLEAU.mkdir(exist_ok=True)

# Gathering the models results
all_models = {
    "Rules": rules_45_5_log,
    "Static LR (9)": static_lr_log,
    "Adaptive LR (9)": adaptive_lr_log,
    "Static LightGBM (9)": static_lgbm_log,
    "Adaptive LightGBM (9)": adaptive_lgbm_log,
    "Static LR (11)": stage_b_logs["Static LR"],
    "Adaptive LR (11)": stage_b_logs["Adaptive LR"],
    "Static LightGBM (11)": stage_b_logs["Static LightGBM"],
    "Adaptive LightGBM (11)": stage_b_logs["Adaptive LightGBM"]}

# Calculating the final metrics
results = []

for name, log in all_models.items():
    caught = int(log["label"].sum())
    reviews = len(log)
    precision = round(caught / reviews * 100, 2)
    recall = round(caught / deployment["label"].sum() * 100, 2)

    results.append([name, reviews, caught, precision, recall])

model_summary = pd.DataFrame(
    results,
    columns=["Model", "Reviews", "Caught", "Precision (%)", "Recall (%)"])

# Saving for Tableau
model_summary.to_csv(TABLEAU / "model_comparison.csv", index=False)

display(model_summary)
print("Saved: model_comparison.csv")


,Model,Reviews,Caught,Precision (%),Recall (%)
0,Rules,11750,163,1.39,2.38
1,Static LR (9),11750,31,0.26,0.45
2,Adaptive LR (9),11750,414,3.52,6.03
3,Static LightGBM (9),11750,225,1.91,3.28
4,Adaptive LightGBM (9),11750,455,3.87,6.63
5,Static LR (11),11750,33,0.28,0.48
6,Adaptive LR (11),11750,451,3.84,6.57
7,Static LightGBM (11),11750,254,2.16,3.70
8,Adaptive LightGBM (11),11750,716,6.09,10.44


Saved: model_comparison.csv


Now ill export the monthly results for each model so I can compare how detection changed over time and show the effects of the feedback loop in Tableau.

In [86]:
# Total laundering cases in each month
monthly_total = deployment.groupby(
    deployment["Date"].dt.to_period("M"))["label"].sum()

# Gathering monthly results for each model
monthly_rows = []

for name, log in all_models.items():
    monthly = log.groupby(
        log["Date"].dt.to_period("M"))["label"].agg(["sum", "count"])
    monthly["Model"] = name
    monthly["Total Cases"] = monthly_total
    monthly_rows.append(monthly.reset_index())

# Combining the results and calculating monthly recall
monthly_results = pd.concat(monthly_rows, ignore_index=True)
monthly_results.columns = ["Month", "Caught", "Reviews", "Model", "Total Cases"]

monthly_results["Recall (%)"] = (
    monthly_results["Caught"] / monthly_results["Total Cases"] * 100).round(2)

monthly_results["Month"] = monthly_results["Month"].astype(str)

# Exporting to Tableau
monthly_results.to_csv(TABLEAU / "monthly_performance.csv", index=False)

display(monthly_results.head(12))
print("Saved: monthly_performance.csv")


,Month,Caught,Reviews,Model,Total Cases,Recall (%)
0,2023-01,15,1550,Rules,880,1.70
1,2023-02,19,1400,Rules,888,2.14
2,2023-03,18,1550,Rules,849,2.12
3,2023-04,20,1500,Rules,888,2.25
4,2023-05,17,1550,Rules,789,2.15
5,2023-06,29,1500,Rules,913,3.18
6,2023-07,32,1550,Rules,904,3.54
7,2023-08,13,1150,Rules,750,1.73
8,2023-01,6,1550,Static LR (9),880,0.68
9,2023-02,4,1400,Static LR (9),888,0.45


Saved: monthly_performance.csv


Next i will export the laundering type results from Part E so I can compare how the models performed across different types of laundering in Tableau.


In [88]:
# Collecting detections for every model and laundering type
rows = []

for name, log in all_models.items():
    caught = log.loc[log["label"] == 1, "Laundering_type"].value_counts()

    for laundering_type, total in total_types.items():
        rows.append([
            name, laundering_type, total,
            caught.get(laundering_type, 0)])

typology_export = pd.DataFrame(
    rows,
    columns=["Model", "Laundering_type", "Total Cases", "Caught"])

# Adding information for Tableau filters
typology_export["Training"] = typology_export["Model"].str.split().str[0]

typology_export["Features"] = typology_export["Model"].str.extract(
    r"\((9|11)\)")[0].fillna("Rules")

typology_export["Network Related"] = (
    typology_export["Laundering_type"].isin(network_types))

typology_export["Recall (%)"] = (
    typology_export["Caught"] / typology_export["Total Cases"] * 100).round(2)

# Exporting the complete dataset
typology_export.to_csv(TABLEAU / "typology_performance.csv", index=False)

display(typology_export)
print("Saved:", len(typology_export), "rows")


,Model,Laundering_type,Total Cases,Caught,Training,Features,Network Related,Recall (%)
0,Rules,structuring,1373,6,Rules,Rules,False,0.44
1,Rules,cash_withdrawal,801,1,Rules,Rules,False,0.12
2,Rules,deposit_send,732,22,Rules,Rules,False,3.01
3,Rules,smurfing,710,2,Rules,Rules,False,0.28
4,Rules,layered_fan_in,528,7,Rules,Rules,True,1.33
...,...,...,...,...,...,...,...,...
148,Adaptive LightGBM (11),behavioural_change_2,214,175,Adaptive,11,False,81.78
149,Adaptive LightGBM (11),behavioural_change_1,206,134,Adaptive,11,False,65.05
150,Adaptive LightGBM (11),single_large,172,7,Adaptive,11,False,4.07
151,Adaptive LightGBM (11),fan_out,128,3,Adaptive,11,True,2.34


Saved: 153 rows


Next i will export the capacity and labor savings results from Part F so I can use them in Tableau.

In [89]:

# Export daily review limits and detection results
capacity_results.to_csv(
    TABLEAU / "review_capacity.csv", index=False
)

# Export estimated investigator time and labor savings
time_savings.to_csv(
    TABLEAU / "labor_savings.csv", index=False
)

print("Saved: review_capacity.csv")
print("Saved: labor_savings.csv")


Saved: review_capacity.csv
Saved: labor_savings.csv


Finally, I'll check that all five CSV files were saved correctly and are ready to use in Tableau.

In [90]:

# Check all Tableau exports
files = [
    "model_comparison.csv",
    "monthly_performance.csv",
    "typology_performance.csv",
    "review_capacity.csv",
    "labor_savings.csv"
]

for file in files:
    data = pd.read_csv(TABLEAU / file)
    print(file, "-", len(data), "rows")


model_comparison.csv - 9 rows
monthly_performance.csv - 72 rows
typology_performance.csv - 153 rows
review_capacity.csv - 11 rows
labor_savings.csv - 4 rows
